# Practice: Explain the Code — Valuing a Private Loan

## Background
 The fastest-growing corner of modern finance — **private credit**, loans negotiated directly between a fund and a company — rarely writes terms that clean. A realistic term sheet line reads:

> *"€100M unitranche, 5-year bullet, **6% cash + 4% PIK**, issued at **97 (3 points OID)**."*

Two instruments of the trade hide in that sentence:

- **OID — original issue discount.** The lender hands over less than face value (here 97 for a face of 100) but interest and repayment run on the full face. Economically it is yield dressed up as a discount — and often a fee dressed up as yield.
- **PIK — payment-in-kind.** Instead of paying interest in cash, the borrower pays it *in more debt*: the interest is added to the balance, which then itself earns interest. The lender books income every year; the cash arrives — if it arrives — at the very end.

Why do these exist? OID lets arrangers take economics upfront without raising the headline coupon; PIK lets a cash-strapped borrower (a freshly leveraged LBO company, say) conserve cash in the lean years. Both are everywhere in direct lending — and both drive a wedge between the **stated coupon**, the **promised yield**, and the **realized return**. Today's forty lines of code: a loan that pays 8% cash can yield over 10%, and a loan that accrues 10% on paper can realize exactly 0%.


## Problem

A credit fund has lent money to a company under the following contract:

- The loan's **face value is €100M** — but the fund actually **advanced only €97M**.
- Each year, the borrower pays **6% of the current loan balance in cash**.
- Another **4% per year is not paid in cash** — it is **added to the balance**, which therefore grows over time (and next year's interest runs on the bigger balance).
- After **5 years**, the borrower repays the entire (grown) balance in one payment.

*(In industry jargon this is a "5-year unitranche, 6% cash + 4% PIK, issued at 97.")*

Question: *what rate of return does this contract promise*, and *what is the loan worth today*, given where market interest rates are right now?

**Step 1: load the tools.**

- `numpy` (`np`): arrays and vectorized maths. Here it builds the timeline and the cash flows, and discounts them all at once.
- `pandas` (`pd`): tables with dates. Here it reads the Treasury yields from FRED.
- `fsolve` (from `scipy.optimize`): finds the value of a variable that makes a function equal to zero. We use it to find the IRR, the rate at which the NPV is zero.
- `matplotlib.pyplot` (`plt`): plots. Imported but not used in this notebook.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import fsolve
import matplotlib.pyplot as plt

**Step 2: download today's 5-year Treasury yield.**

- `GS5` is the FRED code for the yield on 5-year US government bonds. It is the risk-free rate for a 5-year horizon, which matches the loan's maturity.
- The `fredgraph.csv` link downloads the series as a CSV file, so no API key is needed.
- `pd.read_csv(url, ...)` reads it straight into a DataFrame:
  - `index_col=0` makes the first column (the dates) the index.
  - `parse_dates=True` turns those dates into real dates instead of text.
  - `.dropna()` removes months with missing values.
- The result, `rates`, is a table with one row per month and one column of yields in percent (e.g. `4.93`).

In [2]:
# GS5: Market Yield on U.S. Treasury Securities at 5-Year Constant Maturity
url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GS5'
rates = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

**Step 3: the return the market requires for this loan.**

- `rates.iloc[-1, 0]` takes the **last row** (`-1`, the most recent month) and the **first column** (`0`). `iloc` selects by position.
- `/ 100`: FRED quotes yields in percent, so 4.93 becomes 0.0493.
- `credit_spread = 0.045`: lending to a private, leveraged company is riskier than lending to the US government, so investors demand 4.5 percentage points more.
- `r_req = r_f + credit_spread`: the **required yield**, about 4.93% + 4.5% = 9.43%. This is the discount rate used to value the loan.

In [3]:
r_f = rates.iloc[-1, 0] / 100
credit_spread = 0.045
r_req = r_f + credit_spread

**Step 4: the contract terms (in € millions).**

| Variable | Value | Meaning |
|---|---|---|
| `principal` | 100 | face value: interest and repayment are calculated on 100 |
| `OID` | 0.03 | original issue discount: the fund only pays 100 × (1 − 0.03) = **97** |
| `cash_rate` | 0.06 | 6% of the balance is paid **in cash** every year |
| `pik_rate` | 0.04 | 4% of the balance is **added to the loan** every year instead of being paid (payment-in-kind) |
| `maturity` | 5 | the whole balance is repaid after 5 years |

Writing the inputs at the top as named variables, instead of typing the numbers inside the formulas, means you can change one assumption and rerun everything.

In [4]:
principal = 100
OID = 0.03
cash_rate = 0.06
pik_rate = 0.04
maturity = 5

**Step 5: the loan balance in each year.**

- `t = np.arange(1, maturity+1)` gives the years `[1, 2, 3, 4, 5]`. `arange` stops *before* its end value, hence `maturity+1`.
- Each year the 4% PIK is added to the balance, so the balance grows like compound interest: `principal * (1+pik_rate)**(t-1)`.
- `t - 1` because the interest paid in year *t* is calculated on the balance at the **start** of that year. In year 1 nothing has been added yet.
- Because `t` is an array, the whole formula is computed for all 5 years at once (vectorized, no loop).

| Year | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Balance at start of year | 100.00 | 104.00 | 108.16 | 112.49 | 116.99 |

In [5]:
t = np.arange(1, maturity+1)
loan_balances = principal * (1+pik_rate)**(t - 1)

**Step 6: the fund's cash flows, from t = 0 to t = 5.**

- `flows = np.zeros(maturity+1)` makes 6 slots, one per date from today (t = 0) to year 5.
- `flows[0] = -principal*(1-OID)`: today the fund **pays out** 97. Negative because money leaves the fund.
- `flows[1:] = cash_rate * loan_balances`: in years 1 to 5 the fund receives 6% of that year's balance in cash. `[1:]` means "from position 1 to the end".
- `flows[-1] += principal * (1+pik_rate)**maturity`: in the last year (`[-1]`) the borrower also repays the grown balance, 100 × 1.04⁵ = 121.67. `+=` **adds** it to the interest already in that slot instead of replacing it.

| t | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| Cash flow | −97.00 | 6.00 | 6.24 | 6.49 | 6.75 | 7.02 + 121.67 = 128.69 |

In [6]:
flows = np.zeros(maturity+1)
flows[0] = -principal*(1-OID)
flows[1:] = cash_rate * loan_balances
flows[-1] += principal * (1+pik_rate)**maturity

**Step 7: the promised yield (the IRR).**

- `times = np.arange(len(flows))` gives the date of each cash flow: `[0, 1, 2, 3, 4, 5]`.
- `lambda r: np.sum(flows / (1 + r)**times)` is a small function of `r`: the **NPV** of the cash flows at rate `r`. At t = 0 the divisor is 1, so the −97 is not discounted.
- `fsolve(function, 0.10)` searches for the `r` where that NPV equals **zero**, starting from a guess of 10%. That rate is the IRR, the return the contract promises if everything is paid as agreed.
- `fsolve` returns an array, so `[0]` takes the number out of it.
- **Result:** about 10.75%. That is above the 10% total coupon (6% cash + 4% PIK) because the fund paid only 97 but earns interest and repayment on 100.

In [7]:
times = np.arange(len(flows))
y = fsolve(lambda r: np.sum(flows / (1 + r)**times), 0.10)[0]

**Step 8: what the loan is worth today.**

- This is a present value: each future cash flow divided by `(1 + r_req)` raised to its date, then added up.
- `flows[1:]` and `times[1:]` skip t = 0. The value is what the **future** payments are worth today, so the 97 paid out is left out.
- The discount rate is `r_req` (about 9.43%), the return the market requires, **not** the loan's own yield. Discounting at the loan's own IRR would give exactly 97, by definition.
- **Result:** about 102.36. The loan promises more (10.75%) than the market requires (9.43%), so it is worth more than the 97 the fund paid.

In [8]:
value = np.sum(flows[1:] / (1 + r_req)**times[1:])

**Step 9: print the results.**

- `f"..."` strings insert the variables into the text.
- `{r_f:.2%}` formats a number as a percentage with 2 decimals (0.0493 → `4.93%`).
- `{value:.2f}` shows a plain number with 2 decimals.

**How to read the output:** the contract promises 10.75%, the market requires 9.43%, so the loan is worth about 102 for a price of 97. The Treasury yield, and so the value, changes with the day you run the notebook.

**The catch:** that 10.75% is a *promised* yield. The 4% PIK interest arrives as cash only in year 5, inside the final 121.67, and only if the borrower can pay. If the borrower defaults, the PIK interest the fund booked as income every year is never received. The **realized** return can be far below the promised one.

In [9]:
print(f"5-year Treasury yield today: {r_f:.2%}  ->  required yield: {r_req:.2%}")
print(f"Promised yield of the loan:  {y:.2%}")
print(f"Value of the loan today:     {value:.2f}   (the fund paid 97.00)")

5-year Treasury yield today: 4.93%  ->  required yield: 9.43%
Promised yield of the loan:  10.75%
Value of the loan today:     102.36   (the fund paid 97.00)
